# Project 5 — Neural Networks for Molecular Prediction

This notebook extends the earlier descriptor and fingerprint projects into neural-network workflows for molecular property prediction.

The notebook compares three modelling routes:

1. **Fingerprint MLP** — Morgan fingerprints are treated as fixed-length numerical vectors and passed into a PyTorch multilayer perceptron.
2. **Graph neural network** — molecules are represented as atom-and-bond graphs and processed using PyTorch Geometric.
3. **DeepChem workflow** — DeepChem is used for molecular featurisation and model wrapping, then compared with the PyTorch workflow.

The property used here is aqueous solubility, expressed as `logS`. When internet access is available, the notebook loads a public solubility dataset. If the network is unavailable, it falls back to a small built-in dataset so the notebook can still run.

## 1. Setup

This section imports the scientific Python stack, fixes random seeds and defines a few helper functions. Reproducibility matters because neural-network training is sensitive to random initialisation, train/test split and mini-batch order.

In [ ]:
import os
import math
import random
import warnings
from io import BytesIO
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem, Draw

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)
from sklearn.ensemble import RandomForestRegressor

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Silence noisy RDKit parsing warnings for invalid demonstration SMILES.
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

# Keep CPU training predictable on normal laptops.
os.environ["OMP_NUM_THREADS"] = "2"
torch.set_num_threads(2)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

### Exercise — Check the runtime setup

Print the versions of the core libraries and check whether PyTorch can see a GPU. This is useful when the same notebook is run locally, on Kaggle, or in a Conda environment.

In [ ]:
RUN_THIS_EXERCISE = False

if RUN_THIS_EXERCISE:
    # TODO: import each package you want to inspect.
    import pandas as ____
    import sklearn as ____

    # TODO: print package versions.
    print("pandas:", ____)
    print("scikit-learn:", ____)

    # TODO: check the PyTorch device.
    print("PyTorch device:", ____)

### Answer — Check the runtime setup

In [ ]:
# Print versions that often affect notebook behaviour.
import sklearn

print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", DEVICE)

## 2. Public solubility data with fallback

The main dataset path tries to read AqSolDB-style solubility data from a public CSV. The fallback dataset keeps the notebook usable without internet access. After loading, the dataset is capped to at most 500 molecules using random sampling to avoid slow descriptor, fingerprint and neural-network sections.

In [ ]:
def make_demo_solubility_data():
    records = [
        {"name": "methanol", "smiles": "CO", "target_logS": 0.10},
        {"name": "ethanol", "smiles": "CCO", "target_logS": -0.20},
        {"name": "propanol", "smiles": "CCCO", "target_logS": -0.70},
        {"name": "butanol", "smiles": "CCCCO", "target_logS": -1.10},
        {"name": "pentanol", "smiles": "CCCCCO", "target_logS": -1.70},
        {"name": "hexanol", "smiles": "CCCCCCO", "target_logS": -2.20},
        {"name": "acetone", "smiles": "CC(=O)C", "target_logS": -0.20},
        {"name": "acetic acid", "smiles": "CC(=O)O", "target_logS": 0.10},
        {"name": "ethyl acetate", "smiles": "CCOC(=O)C", "target_logS": -0.70},
        {"name": "diethyl ether", "smiles": "CCOCC", "target_logS": -1.00},
        {"name": "benzene", "smiles": "c1ccccc1", "target_logS": -2.10},
        {"name": "toluene", "smiles": "Cc1ccccc1", "target_logS": -2.70},
        {"name": "phenol", "smiles": "Oc1ccccc1", "target_logS": -1.50},
        {"name": "aniline", "smiles": "Nc1ccccc1", "target_logS": -1.30},
        {"name": "benzoic acid", "smiles": "O=C(O)c1ccccc1", "target_logS": -1.80},
        {"name": "benzaldehyde", "smiles": "O=Cc1ccccc1", "target_logS": -1.90},
        {"name": "benzyl alcohol", "smiles": "OCc1ccccc1", "target_logS": -1.10},
        {"name": "vanillin", "smiles": "COc1cc(C=O)ccc1O", "target_logS": -1.20},
        {"name": "cinnamaldehyde", "smiles": "O=CC=Cc1ccccc1", "target_logS": -2.30},
        {"name": "limonene", "smiles": "CC1=CCC(CC1)C(=C)C", "target_logS": -4.30},
        {"name": "linalool", "smiles": "CC(C)=CCCC(C)(O)C=C", "target_logS": -2.80},
        {"name": "menthol", "smiles": "CC(C)C1CCC(C)CC1O", "target_logS": -3.20},
        {"name": "aspirin", "smiles": "CC(=O)OC1=CC=CC=C1C(=O)O", "target_logS": -2.30},
        {"name": "caffeine", "smiles": "Cn1cnc2c1c(=O)n(C)c(=O)n2C", "target_logS": -1.00},
        {"name": "paracetamol", "smiles": "CC(=O)NC1=CC=C(O)C=C1", "target_logS": -1.40},
        {"name": "ibuprofen", "smiles": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", "target_logS": -4.00},
        {"name": "naphthalene", "smiles": "c1ccc2ccccc2c1", "target_logS": -3.60},
        {"name": "anthracene", "smiles": "c1ccc2cc3ccccc3cc2c1", "target_logS": -5.10},
        {"name": "glucose", "smiles": "C(C1C(C(C(C(O1)O)O)O)O)O", "target_logS": 0.70},
        {"name": "cholesterol", "smiles": "CC(C)CCCC(C)C1CCC2C3CCC4=CC(O)CCC4(C)C3CCC12C", "target_logS": -7.00},
    ]
    return pd.DataFrame(records)


def read_public_csv_with_timeout(url, timeout=12):
    # urlopen adds a timeout so an offline notebook does not hang.
    with urlopen(url, timeout=timeout) as response:
        return pd.read_csv(BytesIO(response.read()))


def standardise_solubility_columns(raw_df):
    # Accept common public-dataset column names and convert them to one notebook schema.
    column_map_options = [
        {"Name": "name", "SMILES": "smiles", "Solubility": "target_logS"},
        {"Name": "name", "SMILES": "smiles", "Solubility_logS": "target_logS"},
        {"Compound ID": "name", "smiles": "smiles", "measured log solubility in mols per litre": "target_logS"},
        {"name": "name", "smiles": "smiles", "target_logS": "target_logS"},
    ]

    for column_map in column_map_options:
        if set(column_map).issubset(raw_df.columns):
            df = raw_df[list(column_map)].rename(columns=column_map).copy()
            df["target_logS"] = pd.to_numeric(df["target_logS"], errors="coerce")
            df = df.dropna(subset=["smiles", "target_logS"]).copy()
            df["name"] = df["name"].fillna("unknown")
            return df

    raise ValueError(f"Could not find usable solubility columns. Columns found: {list(raw_df.columns)}")


def load_public_or_demo_solubility_data(max_molecules=500, random_state=42):
    urls = [
        "https://raw.githubusercontent.com/mcsorkun/AqSolDB/master/results/data_curated.csv",
        "https://raw.githubusercontent.com/PatWalters/chem_tutorial/main/curated-solubility-dataset.csv",
    ]

    for url in urls:
        try:
            raw_df = read_public_csv_with_timeout(url)
            df = standardise_solubility_columns(raw_df)
            data_source = f"public dataset: {url}"
            break
        except Exception as error:
            print("Public load failed:", error)
    else:
        df = make_demo_solubility_data()
        data_source = "built-in demo dataset"

    # Randomly cap the dataset so all downstream sections remain laptop-friendly.
    if len(df) > max_molecules:
        df = df.sample(n=max_molecules, random_state=random_state).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    return df, data_source


MAX_MOLECULES = 500
df, data_source = load_public_or_demo_solubility_data(
    max_molecules=MAX_MOLECULES,
    random_state=SEED,
)

print("Data source:", data_source)
print("Dataset used:", df.shape)
df.head()

In [ ]:
# Check the target range before modelling.
print(df["target_logS"].describe())

# Very soluble examples have higher logS values.
display(df.sort_values("target_logS", ascending=False).head(5))

# Poorly soluble examples have lower logS values.
display(df.sort_values("target_logS", ascending=True).head(5))

In [ ]:
# Visualise the target distribution before model training.
plt.figure(figsize=(7, 5))
plt.hist(df["target_logS"], bins=30, edgecolor="black")

plt.xlabel("logS")
plt.ylabel("Count")
plt.title("Solubility target distribution")
plt.show()

### Exercise — Modify the dataset cap

Change the maximum number of molecules used by the notebook. Try `100`, `300` and `500`, then compare how the target distribution changes.

In [ ]:
RUN_THIS_EXERCISE = False

if RUN_THIS_EXERCISE:
    # TODO: choose a smaller or larger cap.
    max_molecules_to_try = ____

    # TODO: reload the data using your chosen cap.
    exercise_df, exercise_source = load_public_or_demo_solubility_data(
        max_molecules=____,
        random_state=SEED,
    )

    # TODO: print the source and shape.
    print(____)
    print(____)

    # TODO: plot the target distribution.
    plt.figure(figsize=(7, 5))
    plt.hist(____, bins=____, edgecolor="black")
    plt.xlabel(____)
    plt.ylabel(____)
    plt.title(____)
    plt.show()

### Answer — Modify the dataset cap

In [ ]:
# Load a smaller subset to check how the cap changes the working dataset.
exercise_df, exercise_source = load_public_or_demo_solubility_data(
    max_molecules=100,
    random_state=SEED,
)

print("Source:", exercise_source)
print("Shape:", exercise_df.shape)

# Plot the target distribution for the smaller subset.
plt.figure(figsize=(7, 5))
plt.hist(exercise_df["target_logS"], bins=20, edgecolor="black")
plt.xlabel("logS")
plt.ylabel("Count")
plt.title("Target distribution with 100-molecule cap")
plt.show()

## 3. RDKit molecules and train/validation/test split

Neural-network sections reuse the same cleaned molecular table. Invalid SMILES are removed, target values are scaled for neural-network training and a fixed split is used so model comparisons are based on the same molecules.

In [ ]:
# Convert SMILES into RDKit molecule objects.
df["mol"] = df["smiles"].apply(lambda smiles: Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None)

# Keep only molecules RDKit can parse.
valid_df = df[df["mol"].notnull()].copy().reset_index(drop=True)

# Canonical SMILES help remove duplicate structures.
valid_df["canonical_smiles"] = valid_df["mol"].apply(Chem.MolToSmiles)
valid_df = valid_df.drop_duplicates(subset=["canonical_smiles"]).reset_index(drop=True)

print("Valid molecules:", valid_df.shape)
valid_df[["name", "smiles", "canonical_smiles", "target_logS"]].head()

In [ ]:
# Split once so all models are evaluated on the same test molecules.
train_df, temp_df = train_test_split(
    valid_df,
    test_size=0.30,
    random_state=SEED,
)

valid_df_split, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
)

print("Train:", train_df.shape)
print("Valid:", valid_df_split.shape)
print("Test:", test_df.shape)

In [ ]:
# Scale the continuous target for neural-network loss calculation.
target_scaler = StandardScaler()

y_train_scaled = target_scaler.fit_transform(train_df[["target_logS"]]).astype(np.float32)
y_valid_scaled = target_scaler.transform(valid_df_split[["target_logS"]]).astype(np.float32)
y_test_scaled = target_scaler.transform(test_df[["target_logS"]]).astype(np.float32)

# Keep unscaled targets for final MAE, RMSE and R2.
y_train_true = train_df["target_logS"].to_numpy(dtype=np.float32)
y_valid_true = valid_df_split["target_logS"].to_numpy(dtype=np.float32)
y_test_true = test_df["target_logS"].to_numpy(dtype=np.float32)

print("Scaled target mean:", round(float(y_train_scaled.mean()), 4))
print("Scaled target std:", round(float(y_train_scaled.std()), 4))

### Exercise — Inspect split balance

Check whether the train, validation and test sets cover similar logS ranges. Large distribution shifts can make model comparison misleading.

In [ ]:
RUN_THIS_EXERCISE = False

if RUN_THIS_EXERCISE:
    # TODO: build a summary table for train, validation and test targets.
    split_summary = pd.DataFrame({
        "split": [____, ____, ____],
        "n": [____, ____, ____],
        "mean_logS": [____, ____, ____],
        "min_logS": [____, ____, ____],
        "max_logS": [____, ____, ____],
    })

    # TODO: display the summary table.
    display(____)

    # TODO: overlay histograms for the three splits.
    plt.figure(figsize=(7, 5))
    plt.hist(____, bins=20, alpha=0.5, label="train")
    plt.hist(____, bins=20, alpha=0.5, label="valid")
    plt.hist(____, bins=20, alpha=0.5, label="test")
    plt.xlabel("logS")
    plt.ylabel("Count")
    plt.title("Target distribution by split")
    plt.legend()
    plt.show()

### Answer — Inspect split balance

In [ ]:
# Compare target ranges across the three splits.
split_summary = pd.DataFrame({
    "split": ["train", "valid", "test"],
    "n": [len(train_df), len(valid_df_split), len(test_df)],
    "mean_logS": [
        train_df["target_logS"].mean(),
        valid_df_split["target_logS"].mean(),
        test_df["target_logS"].mean(),
    ],
    "min_logS": [
        train_df["target_logS"].min(),
        valid_df_split["target_logS"].min(),
        test_df["target_logS"].min(),
    ],
    "max_logS": [
        train_df["target_logS"].max(),
        valid_df_split["target_logS"].max(),
        test_df["target_logS"].max(),
    ],
})

display(split_summary)

# Overlay split distributions to detect obvious imbalance.
plt.figure(figsize=(7, 5))
plt.hist(train_df["target_logS"], bins=20, alpha=0.5, label="train")
plt.hist(valid_df_split["target_logS"], bins=20, alpha=0.5, label="valid")
plt.hist(test_df["target_logS"], bins=20, alpha=0.5, label="test")
plt.xlabel("logS")
plt.ylabel("Count")
plt.title("Target distribution by split")
plt.legend()
plt.show()

## 4. Morgan fingerprints for the MLP

Morgan fingerprints convert each molecule into a fixed-length vector. The MLP cannot read a SMILES string directly, so the fingerprint acts as a numerical molecular representation.

In [ ]:
def mol_to_morgan_fp(mol, radius=2, n_bits=1024):
    # Convert one RDKit molecule into a binary Morgan fingerprint vector.
    bitvect = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(bitvect, arr)
    return arr


FP_RADIUS = 2
FP_BITS = 1024

# Build fingerprint matrices for each split.
X_train_fp = np.vstack(train_df["mol"].apply(lambda mol: mol_to_morgan_fp(mol, FP_RADIUS, FP_BITS)))
X_valid_fp = np.vstack(valid_df_split["mol"].apply(lambda mol: mol_to_morgan_fp(mol, FP_RADIUS, FP_BITS)))
X_test_fp = np.vstack(test_df["mol"].apply(lambda mol: mol_to_morgan_fp(mol, FP_RADIUS, FP_BITS)))

print("Train fingerprint matrix:", X_train_fp.shape)
print("Validation fingerprint matrix:", X_valid_fp.shape)
print("Test fingerprint matrix:", X_test_fp.shape)

In [ ]:
# Fingerprint density shows how sparse the binary molecular vectors are.
fingerprint_density = X_train_fp.mean(axis=1)

plt.figure(figsize=(7, 5))
plt.hist(fingerprint_density, bins=25, edgecolor="black")
plt.xlabel("Fraction of active fingerprint bits")
plt.ylabel("Molecule count")
plt.title("Morgan fingerprint density")
plt.show()

print("Mean active-bit fraction:", fingerprint_density.mean())

### Exercise — Compare fingerprint sizes

Recalculate fingerprints with a smaller bit length, such as 256 or 512. Smaller fingerprints are faster but can create more bit collisions.

In [ ]:
RUN_THIS_EXERCISE = False

if RUN_THIS_EXERCISE:
    # TODO: choose a smaller fingerprint size.
    exercise_bits = ____

    # TODO: generate fingerprints for the training molecules.
    exercise_fp = np.vstack(
        train_df["mol"].apply(lambda mol: mol_to_morgan_fp(mol, radius=____, n_bits=____))
    )

    # TODO: print the new matrix shape.
    print("Exercise fingerprint matrix:", ____)

    # TODO: calculate and plot active-bit density.
    exercise_density = ____
    plt.figure(figsize=(7, 5))
    plt.hist(____, bins=25, edgecolor="black")
    plt.xlabel("Fraction of active bits")
    plt.ylabel("Molecule count")
    plt.title("Fingerprint density with smaller bit length")
    plt.show()

### Answer — Compare fingerprint sizes

In [ ]:
# Recalculate fingerprints using fewer bits.
exercise_bits = 512
exercise_fp = np.vstack(
    train_df["mol"].apply(lambda mol: mol_to_morgan_fp(mol, radius=2, n_bits=exercise_bits))
)

print("Exercise fingerprint matrix:", exercise_fp.shape)

# Compare sparsity under the smaller representation.
exercise_density = exercise_fp.mean(axis=1)
plt.figure(figsize=(7, 5))
plt.hist(exercise_density, bins=25, edgecolor="black")
plt.xlabel("Fraction of active bits")
plt.ylabel("Molecule count")
plt.title("Fingerprint density with 512 bits")
plt.show()

## 5. Baseline random forest on fingerprints

A classical baseline helps check whether the neural network is adding value. If a simple random forest performs similarly or better, the MLP may be overfitting or under-tuned.

In [ ]:
# Train a non-neural baseline using the same Morgan fingerprints.
rf_baseline = RandomForestRegressor(
    n_estimators=300,
    random_state=SEED,
    n_jobs=-1,
)

rf_baseline.fit(X_train_fp, y_train_true)
rf_pred = rf_baseline.predict(X_test_fp)

rf_mae = mean_absolute_error(y_test_true, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test_true, rf_pred))
rf_r2 = r2_score(y_test_true, rf_pred)

print(f"RF MAE:  {rf_mae:.3f}")
print(f"RF RMSE: {rf_rmse:.3f}")
print(f"RF R2:   {rf_r2:.3f}")

In [ ]:
# Plot random forest predictions for comparison with the MLP later.
plt.figure(figsize=(7, 5))
plt.scatter(y_test_true, rf_pred, alpha=0.7)

# The dashed line marks perfect prediction.
low = min(y_test_true.min(), rf_pred.min())
high = max(y_test_true.max(), rf_pred.max())
plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Observed logS")
plt.ylabel("Predicted logS")
plt.title("Random forest fingerprint baseline")
plt.show()

## 6. PyTorch fingerprint MLP regression

The MLP reads the Morgan fingerprint vector and learns nonlinear combinations of fingerprint bits. The target is scaled during training, then converted back to original logS units for evaluation.

In [ ]:
# Convert fingerprint arrays into PyTorch tensors.
X_train_tensor = torch.tensor(X_train_fp, dtype=torch.float32)
X_valid_tensor = torch.tensor(X_valid_fp, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_fp, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
y_valid_tensor = torch.tensor(y_valid_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

# Mini-batches make training faster and add mild stochasticity.
BATCH_SIZE = 32

train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

valid_loader = DataLoader(
    TensorDataset(X_valid_tensor, y_valid_tensor),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [ ]:
class FingerprintMLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.25):
        super().__init__()

        # Linear layers learn nonlinear combinations of fingerprint bits.
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.network(x)


mlp_model = FingerprintMLPRegressor(
    input_dim=FP_BITS,
    hidden_dim=256,
    dropout=0.25,
).to(DEVICE)

print(mlp_model)

In [ ]:
def train_regression_model(model, train_loader, valid_loader, n_epochs=60, lr=1e-3, patience=8):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    train_losses = []
    valid_losses = []
    best_valid_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        train_loss_total = 0.0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

            train_loss_total += loss.item() * xb.size(0)

        train_loss = train_loss_total / len(train_loader.dataset)

        model.eval()
        valid_loss_total = 0.0

        with torch.no_grad():
            for xb, yb in valid_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)

                pred = model(xb)
                loss = loss_fn(pred, yb)
                valid_loss_total += loss.item() * xb.size(0)

        valid_loss = valid_loss_total / len(valid_loader.dataset)

        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        # Keep the model from the epoch with the best validation loss.
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        print(f"Epoch {epoch:02d} | train={train_loss:.4f} | valid={valid_loss:.4f}")

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model, train_losses, valid_losses


mlp_model, mlp_train_losses, mlp_valid_losses = train_regression_model(
    mlp_model,
    train_loader,
    valid_loader,
    n_epochs=60,
    lr=1e-3,
    patience=8,
)

In [ ]:
# Plot training and validation loss to inspect overfitting.
plt.figure(figsize=(7, 5))
plt.plot(mlp_train_losses, label="train")
plt.plot(mlp_valid_losses, label="valid")

# A widening gap means the model fits training molecules better than validation molecules.
plt.xlabel("Epoch")
plt.ylabel("MSE loss on scaled target")
plt.title("Fingerprint MLP training curve")
plt.legend()
plt.show()

In [ ]:
# Predict test-set logS values and convert them back to original units.
mlp_model.eval()

with torch.no_grad():
    mlp_test_scaled_pred = mlp_model(X_test_tensor.to(DEVICE)).cpu().numpy()

mlp_test_pred = target_scaler.inverse_transform(mlp_test_scaled_pred).ravel()

mlp_mae = mean_absolute_error(y_test_true, mlp_test_pred)
mlp_rmse = np.sqrt(mean_squared_error(y_test_true, mlp_test_pred))
mlp_r2 = r2_score(y_test_true, mlp_test_pred)

print(f"MLP MAE:  {mlp_mae:.3f}")
print(f"MLP RMSE: {mlp_rmse:.3f}")
print(f"MLP R2:   {mlp_r2:.3f}")

In [ ]:
# Compare observed and predicted logS values.
plt.figure(figsize=(7, 5))
plt.scatter(y_test_true, mlp_test_pred, alpha=0.7)

# The dashed diagonal line marks perfect prediction.
low = min(y_test_true.min(), mlp_test_pred.min())
high = max(y_test_true.max(), mlp_test_pred.max())
plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Observed logS")
plt.ylabel("Predicted logS")
plt.title("Fingerprint MLP regression performance")
plt.show()

In [ ]:
# Residuals reveal where the model systematically over- or under-predicts.
mlp_residuals = mlp_test_pred - y_test_true

plt.figure(figsize=(7, 5))
plt.scatter(y_test_true, mlp_residuals, alpha=0.7)
plt.axhline(0, linestyle="--")

plt.xlabel("Observed logS")
plt.ylabel("Prediction residual")
plt.title("Fingerprint MLP residuals")
plt.show()

plt.figure(figsize=(7, 5))
plt.hist(mlp_residuals, bins=25, edgecolor="black")
plt.xlabel("Prediction residual")
plt.ylabel("Count")
plt.title("Fingerprint MLP residual distribution")
plt.show()

The MLP should usually show a positive observed-vs-predicted trend. If the training curve keeps improving while the validation curve stops improving, the model is overfitting. That is useful evidence: the notebook is not just reporting a metric, it is showing the model behaviour.

### Exercise — Tune the fingerprint MLP

Change the hidden size, dropout and learning rate. Record whether the validation curve improves or overfits faster.

In [ ]:
RUN_THIS_EXERCISE = False

if RUN_THIS_EXERCISE:
    # TODO: choose a hidden size, dropout and learning rate.
    exercise_hidden_dim = ____
    exercise_dropout = ____
    exercise_lr = ____

    # TODO: build the model.
    exercise_model = FingerprintMLPRegressor(
        input_dim=____,
        hidden_dim=____,
        dropout=____,
    ).to(DEVICE)

    # TODO: train the model.
    exercise_model, exercise_train_losses, exercise_valid_losses = train_regression_model(
        exercise_model,
        train_loader,
        valid_loader,
        n_epochs=____,
        lr=____,
        patience=____,
    )

    # TODO: plot the training curve.
    plt.figure(figsize=(7, 5))
    plt.plot(____, label="train")
    plt.plot(____, label="valid")
    plt.xlabel("Epoch")
    plt.ylabel("MSE loss")
    plt.title("Exercise MLP training curve")
    plt.legend()
    plt.show()

### Answer — Tune the fingerprint MLP

In [ ]:
# Try a smaller model with stronger dropout.
tuned_mlp_model = FingerprintMLPRegressor(
    input_dim=FP_BITS,
    hidden_dim=128,
    dropout=0.35,
).to(DEVICE)

tuned_mlp_model, exercise_train_losses, exercise_valid_losses = train_regression_model(
    tuned_mlp_model,
    train_loader,
    valid_loader,
    n_epochs=40,
    lr=8e-4,
    patience=6,
)

# Plot the training curve for the tuned model.
plt.figure(figsize=(7, 5))
plt.plot(exercise_train_losses, label="train")
plt.plot(exercise_valid_losses, label="valid")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Exercise MLP training curve")
plt.legend()
plt.show()

## 7. PyTorch fingerprint MLP classification

The same molecular fingerprints can be used for classification. Here the continuous logS target is converted into a median-split class: molecules above the training median are labelled more soluble and molecules below it are labelled less soluble.

In [ ]:
# Create a binary classification label from the training-set median.
classification_threshold = train_df["target_logS"].median()

def make_solubility_class(values, threshold):
    # Class 1 means higher-than-median solubility.
    return (np.asarray(values) >= threshold).astype(np.float32)

y_train_class = make_solubility_class(train_df["target_logS"], classification_threshold)
y_valid_class = make_solubility_class(valid_df_split["target_logS"], classification_threshold)
y_test_class = make_solubility_class(test_df["target_logS"], classification_threshold)

print("Classification threshold:", classification_threshold)
print("Train class counts:", np.bincount(y_train_class.astype(int)))
print("Test class counts:", np.bincount(y_test_class.astype(int)))

In [ ]:
class FingerprintMLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.25):
        super().__init__()

        # The final layer outputs one logit for binary classification.
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


clf_train_loader = DataLoader(
    TensorDataset(
        X_train_tensor,
        torch.tensor(y_train_class, dtype=torch.float32),
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

clf_valid_loader = DataLoader(
    TensorDataset(
        X_valid_tensor,
        torch.tensor(y_valid_class, dtype=torch.float32),
    ),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

clf_model = FingerprintMLPClassifier(input_dim=FP_BITS).to(DEVICE)
print(clf_model)

In [ ]:
def train_classifier_model(model, train_loader, valid_loader, n_epochs=50, lr=1e-3, patience=8):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss()

    train_losses = []
    valid_losses = []
    best_valid_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        train_loss_total = 0.0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss_total += loss.item() * xb.size(0)

        train_loss = train_loss_total / len(train_loader.dataset)

        model.eval()
        valid_loss_total = 0.0

        with torch.no_grad():
            for xb, yb in valid_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)

                logits = model(xb)
                loss = loss_fn(logits, yb)
                valid_loss_total += loss.item() * xb.size(0)

        valid_loss = valid_loss_total / len(valid_loader.dataset)

        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        # Store the best validation checkpoint.
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        print(f"Epoch {epoch:02d} | train={train_loss:.4f} | valid={valid_loss:.4f}")

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model, train_losses, valid_losses


clf_model, clf_train_losses, clf_valid_losses = train_classifier_model(
    clf_model,
    clf_train_loader,
    clf_valid_loader,
    n_epochs=50,
    lr=1e-3,
    patience=8,
)

In [ ]:
# Plot classification loss curves.
plt.figure(figsize=(7, 5))
plt.plot(clf_train_losses, label="train")
plt.plot(clf_valid_losses, label="valid")

plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("Fingerprint MLP classification training curve")
plt.legend()
plt.show()

In [ ]:
# Convert test logits into probabilities and class predictions.
clf_model.eval()

with torch.no_grad():
    test_logits = clf_model(X_test_tensor.to(DEVICE)).cpu().numpy()

test_prob = 1 / (1 + np.exp(-test_logits))
test_pred_class = (test_prob >= 0.5).astype(int)

clf_accuracy = accuracy_score(y_test_class, test_pred_class)
clf_precision = precision_score(y_test_class, test_pred_class, zero_division=0)
clf_recall = recall_score(y_test_class, test_pred_class, zero_division=0)
clf_f1 = f1_score(y_test_class, test_pred_class, zero_division=0)

# ROC-AUC needs both classes in the test split.
clf_roc_auc = roc_auc_score(y_test_class, test_prob) if len(np.unique(y_test_class)) == 2 else np.nan
clf_pr_auc = average_precision_score(y_test_class, test_prob) if len(np.unique(y_test_class)) == 2 else np.nan

print(f"Accuracy:  {clf_accuracy:.3f}")
print(f"Precision: {clf_precision:.3f}")
print(f"Recall:    {clf_recall:.3f}")
print(f"F1:        {clf_f1:.3f}")
print(f"ROC-AUC:   {clf_roc_auc:.3f}")
print(f"PR-AUC:    {clf_pr_auc:.3f}")

In [ ]:
# Confusion matrix shows which class is confused.
cm = confusion_matrix(y_test_class, test_pred_class)

plt.figure(figsize=(5, 5))
plt.imshow(cm)
plt.xticks([0, 1], ["less soluble", "more soluble"])
plt.yticks([0, 1], ["less soluble", "more soluble"])
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Fingerprint MLP classification confusion matrix")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()

In [ ]:
# ROC and precision-recall curves show ranking quality across thresholds.
if len(np.unique(y_test_class)) == 2:
    fpr, tpr, _ = roc_curve(y_test_class, test_prob)
    precision_curve, recall_curve, _ = precision_recall_curve(y_test_class, test_prob)

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("ROC curve")
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.plot(recall_curve, precision_curve)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-recall curve")
    plt.show()
else:
    print("ROC and PR curves skipped because the test split has only one class.")

### Exercise — Change the classification threshold

Use a different percentile instead of the median. A stricter threshold changes class balance and can make the classification problem harder.

In [ ]:
RUN_THIS_EXERCISE = False

if RUN_THIS_EXERCISE:
    # TODO: choose a percentile threshold, for example 60 or 70.
    percentile = ____

    # TODO: calculate the threshold from the training target.
    new_threshold = np.percentile(____, ____)

    # TODO: create new train/test labels.
    new_y_train_class = make_solubility_class(____, ____)
    new_y_test_class = make_solubility_class(____, ____)

    # TODO: inspect class balance.
    print("New threshold:", ____)
    print("Train counts:", ____)
    print("Test counts:", ____)

### Answer — Change the classification threshold

In [ ]:
# Use a stricter 60th percentile threshold.
percentile = 60
new_threshold = np.percentile(train_df["target_logS"], percentile)

new_y_train_class = make_solubility_class(train_df["target_logS"], new_threshold)
new_y_test_class = make_solubility_class(test_df["target_logS"], new_threshold)

print("New threshold:", new_threshold)
print("Train counts:", np.bincount(new_y_train_class.astype(int)))
print("Test counts:", np.bincount(new_y_test_class.astype(int)))

## 8. Molecular graphs for GNNs

A graph model uses atoms as nodes and bonds as edges. This representation keeps molecular connectivity explicit instead of compressing the molecule into a fixed fingerprint vector.

In [ ]:
def atom_features(atom):
    # Simple atom features keep the graph section readable.
    return [
        atom.GetAtomicNum(),
        atom.GetTotalDegree(),
        atom.GetFormalCharge(),
        int(atom.GetHybridization()),
        int(atom.GetIsAromatic()),
        atom.GetTotalNumHs(),
    ]


def mol_to_graph_arrays(mol):
    node_features = [atom_features(atom) for atom in mol.GetAtoms()]
    edge_pairs = []

    # Add both directions because most GNN layers use directed edge indices.
    for bond in mol.GetBonds():
        start = bond.GetBeginAtomIdx()
        end = bond.GetEndAtomIdx()
        edge_pairs.append([start, end])
        edge_pairs.append([end, start])

    if len(edge_pairs) == 0:
        edge_pairs = [[0, 0]]

    return np.array(node_features, dtype=np.float32), np.array(edge_pairs, dtype=np.int64).T


# Inspect one molecule as graph arrays before using PyTorch Geometric.
example_mol = train_df.iloc[0]["mol"]
example_x, example_edge_index = mol_to_graph_arrays(example_mol)

print("Example molecule:", train_df.iloc[0]["name"])
print("Node feature matrix:", example_x.shape)
print("Edge index matrix:", example_edge_index.shape)
example_x[:5]

### Exercise — Inspect graph size

Choose three molecules and print their number of atoms and graph edges. Larger molecules usually produce larger graph objects.

In [ ]:
RUN_THIS_EXERCISE = False

if RUN_THIS_EXERCISE:
    # TODO: choose molecule row indices.
    row_indices = [____, ____, ____]

    for idx in row_indices:
        # TODO: get molecule and name.
        mol = ____
        name = ____

        # TODO: convert molecule to graph arrays.
        x_array, edge_index_array = ____

        # TODO: print graph dimensions.
        print(name, "nodes:", ____, "directed edges:", ____)

### Answer — Inspect graph size

In [ ]:
# Inspect graph dimensions for the first three training molecules.
row_indices = [0, 1, 2]

for idx in row_indices:
    mol = train_df.iloc[idx]["mol"]
    name = train_df.iloc[idx]["name"]

    x_array, edge_index_array = mol_to_graph_arrays(mol)

    print(name, "nodes:", x_array.shape[0], "directed edges:", edge_index_array.shape[1])

## 9. Graph neural network regression with PyTorch Geometric

This section builds a graph dataset and trains a small graph neural network. The code is explicit rather than hidden: if PyTorch Geometric is installed, the section runs. If it is not installed, the notebook prints the install command and continues without crashing.

In [ ]:
try:
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader as GeoDataLoader
    from torch_geometric.nn import GCNConv, global_mean_pool

    HAS_PYG = True
    print("PyTorch Geometric is available.")
except Exception as error:
    HAS_PYG = False
    print("PyTorch Geometric is not available.")
    print("Install example:")
    print("pip install torch-geometric")
    print("Import error:", error)

In [ ]:
if HAS_PYG:
    def mol_to_pyg_data(mol, target_scaled):
        x_array, edge_index_array = mol_to_graph_arrays(mol)

        # Node features describe atoms; edge_index describes bonds.
        x = torch.tensor(x_array, dtype=torch.float32)
        edge_index = torch.tensor(edge_index_array, dtype=torch.long)
        y = torch.tensor([target_scaled], dtype=torch.float32)

        return Data(x=x, edge_index=edge_index, y=y)

    graph_train_data = [
        mol_to_pyg_data(mol, target)
        for mol, target in zip(train_df["mol"], y_train_scaled.ravel())
    ]

    graph_valid_data = [
        mol_to_pyg_data(mol, target)
        for mol, target in zip(valid_df_split["mol"], y_valid_scaled.ravel())
    ]

    graph_test_data = [
        mol_to_pyg_data(mol, target)
        for mol, target in zip(test_df["mol"], y_test_scaled.ravel())
    ]

    graph_train_loader = GeoDataLoader(graph_train_data, batch_size=32, shuffle=True)
    graph_valid_loader = GeoDataLoader(graph_valid_data, batch_size=32, shuffle=False)
    graph_test_loader = GeoDataLoader(graph_test_data, batch_size=32, shuffle=False)

    print("Graph train molecules:", len(graph_train_data))
    print("First graph:", graph_train_data[0])
else:
    graph_train_data = []
    graph_valid_data = []
    graph_test_data = []

In [ ]:
if HAS_PYG:
    class MolecularGCNRegressor(nn.Module):
        def __init__(self, node_feature_dim, hidden_dim=64, dropout=0.20):
            super().__init__()

            # GCN layers pass information between connected atoms.
            self.conv1 = GCNConv(node_feature_dim, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, hidden_dim)
            self.dropout = nn.Dropout(dropout)
            self.output = nn.Linear(hidden_dim, 1)

        def forward(self, data):
            x, edge_index, batch = data.x, data.edge_index, data.batch

            x = self.conv1(x, edge_index)
            x = torch.relu(x)
            x = self.dropout(x)

            x = self.conv2(x, edge_index)
            x = torch.relu(x)

            # Pool atom embeddings into one molecule embedding.
            x = global_mean_pool(x, batch)

            return self.output(x).squeeze(1)


    gnn_model = MolecularGCNRegressor(
        node_feature_dim=graph_train_data[0].x.shape[1],
        hidden_dim=64,
        dropout=0.20,
    ).to(DEVICE)

    print(gnn_model)
else:
    gnn_model = None

In [ ]:
def train_gnn_regressor(model, train_loader, valid_loader, n_epochs=50, lr=1e-3, patience=8):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    train_losses = []
    valid_losses = []
    best_valid_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        train_loss_total = 0.0

        for batch in train_loader:
            batch = batch.to(DEVICE)

            optimizer.zero_grad()
            pred = model(batch)
            loss = loss_fn(pred, batch.y.view(-1))
            loss.backward()
            optimizer.step()

            train_loss_total += loss.item() * batch.num_graphs

        train_loss = train_loss_total / len(train_loader.dataset)

        model.eval()
        valid_loss_total = 0.0

        with torch.no_grad():
            for batch in valid_loader:
                batch = batch.to(DEVICE)
                pred = model(batch)
                loss = loss_fn(pred, batch.y.view(-1))
                valid_loss_total += loss.item() * batch.num_graphs

        valid_loss = valid_loss_total / len(valid_loader.dataset)

        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        print(f"Epoch {epoch:02d} | train={train_loss:.4f} | valid={valid_loss:.4f}")

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model, train_losses, valid_losses


if HAS_PYG:
    gnn_model, gnn_train_losses, gnn_valid_losses = train_gnn_regressor(
        gnn_model,
        graph_train_loader,
        graph_valid_loader,
        n_epochs=50,
        lr=1e-3,
        patience=8,
    )
else:
    gnn_train_losses = []
    gnn_valid_losses = []

In [ ]:
if HAS_PYG:
    # Plot the GNN training curve.
    plt.figure(figsize=(7, 5))
    plt.plot(gnn_train_losses, label="train")
    plt.plot(gnn_valid_losses, label="valid")

    plt.xlabel("Epoch")
    plt.ylabel("MSE loss on scaled target")
    plt.title("GNN training curve")
    plt.legend()
    plt.show()
else:
    print("GNN plot skipped because PyTorch Geometric is not installed.")

In [ ]:
if HAS_PYG:
    gnn_model.eval()
    gnn_scaled_preds = []

    with torch.no_grad():
        for batch in graph_test_loader:
            batch = batch.to(DEVICE)
            pred = gnn_model(batch)
            gnn_scaled_preds.extend(pred.cpu().numpy())

    gnn_test_pred = target_scaler.inverse_transform(
        np.array(gnn_scaled_preds).reshape(-1, 1)
    ).ravel()

    gnn_mae = mean_absolute_error(y_test_true, gnn_test_pred)
    gnn_rmse = np.sqrt(mean_squared_error(y_test_true, gnn_test_pred))
    gnn_r2 = r2_score(y_test_true, gnn_test_pred)

    print(f"GNN MAE:  {gnn_mae:.3f}")
    print(f"GNN RMSE: {gnn_rmse:.3f}")
    print(f"GNN R2:   {gnn_r2:.3f}")

    plt.figure(figsize=(7, 5))
    plt.scatter(y_test_true, gnn_test_pred, alpha=0.7)

    low = min(y_test_true.min(), gnn_test_pred.min())
    high = max(y_test_true.max(), gnn_test_pred.max())
    plt.plot([low, high], [low, high], linestyle="--")

    plt.xlabel("Observed logS")
    plt.ylabel("Predicted logS")
    plt.title("GNN regression performance")
    plt.show()
else:
    gnn_mae = np.nan
    gnn_rmse = np.nan
    gnn_r2 = np.nan
    gnn_test_pred = None

### Exercise — Change the GNN hidden dimension

Increase or decrease the GNN hidden dimension and compare the validation curve. This changes the size of atom and molecule embeddings.

In [ ]:
RUN_THIS_EXERCISE = False

if RUN_THIS_EXERCISE and HAS_PYG:
    # TODO: choose a hidden dimension.
    exercise_gnn_hidden = ____

    # TODO: build a new GNN model.
    exercise_gnn = MolecularGCNRegressor(
        node_feature_dim=____,
        hidden_dim=____,
        dropout=____,
    ).to(DEVICE)

    # TODO: train the GNN.
    exercise_gnn, exercise_gnn_train, exercise_gnn_valid = train_gnn_regressor(
        exercise_gnn,
        graph_train_loader,
        graph_valid_loader,
        n_epochs=____,
        lr=____,
        patience=____,
    )

    # TODO: plot the training curve.
    plt.figure(figsize=(7, 5))
    plt.plot(____, label="train")
    plt.plot(____, label="valid")
    plt.xlabel("Epoch")
    plt.ylabel("MSE loss")
    plt.title("Exercise GNN training curve")
    plt.legend()
    plt.show()

### Answer — Change the GNN hidden dimension

In [ ]:
if HAS_PYG:
    # Train a smaller GNN for comparison.
    exercise_gnn = MolecularGCNRegressor(
        node_feature_dim=graph_train_data[0].x.shape[1],
        hidden_dim=32,
        dropout=0.25,
    ).to(DEVICE)

    exercise_gnn, exercise_gnn_train, exercise_gnn_valid = train_gnn_regressor(
        exercise_gnn,
        graph_train_loader,
        graph_valid_loader,
        n_epochs=30,
        lr=1e-3,
        patience=6,
    )

    plt.figure(figsize=(7, 5))
    plt.plot(exercise_gnn_train, label="train")
    plt.plot(exercise_gnn_valid, label="valid")
    plt.xlabel("Epoch")
    plt.ylabel("MSE loss")
    plt.title("Exercise GNN training curve")
    plt.legend()
    plt.show()
else:
    print("Install PyTorch Geometric to run this answer.")

## 10. DeepChem molecular featurisation and regression

DeepChem provides chemistry-specific featurisers and model wrappers. This section uses `CircularFingerprint` to create molecular features, then trains a random forest through a DeepChem `SklearnModel` wrapper.

In [ ]:
try:
    import deepchem as dc
    from deepchem.models import SklearnModel

    HAS_DEEPCHEM = True
    print("DeepChem is available:", dc.__version__)
except Exception as error:
    HAS_DEEPCHEM = False
    print("DeepChem is not available.")
    print("Install example:")
    print("pip install deepchem")
    print("Import error:", error)

In [ ]:
if HAS_DEEPCHEM:
    # DeepChem featuriser creates circular fingerprints from SMILES.
    dc_featurizer = dc.feat.CircularFingerprint(size=1024, radius=2)

    dc_train_X = np.asarray(dc_featurizer.featurize(train_df["smiles"].tolist()), dtype=np.float32)
    dc_valid_X = np.asarray(dc_featurizer.featurize(valid_df_split["smiles"].tolist()), dtype=np.float32)
    dc_test_X = np.asarray(dc_featurizer.featurize(test_df["smiles"].tolist()), dtype=np.float32)

    # DeepChem datasets keep features, labels and molecule IDs together.
    dc_train_dataset = dc.data.NumpyDataset(
        X=dc_train_X,
        y=y_train_true.reshape(-1, 1),
        ids=train_df["smiles"].tolist(),
    )

    dc_test_dataset = dc.data.NumpyDataset(
        X=dc_test_X,
        y=y_test_true.reshape(-1, 1),
        ids=test_df["smiles"].tolist(),
    )

    print("DeepChem train features:", dc_train_X.shape)
    print("DeepChem test features:", dc_test_X.shape)
else:
    dc_train_X = None
    dc_valid_X = None
    dc_test_X = None

In [ ]:
if HAS_DEEPCHEM:
    # Wrap a scikit-learn random forest using DeepChem's model interface.
    dc_rf = SklearnModel(
        RandomForestRegressor(
            n_estimators=300,
            random_state=SEED,
            n_jobs=-1,
        )
    )

    dc_rf.fit(dc_train_dataset)
    dc_rf_pred = dc_rf.predict(dc_test_dataset).ravel()

    deepchem_rf_mae = mean_absolute_error(y_test_true, dc_rf_pred)
    deepchem_rf_rmse = np.sqrt(mean_squared_error(y_test_true, dc_rf_pred))
    deepchem_rf_r2 = r2_score(y_test_true, dc_rf_pred)

    print(f"DeepChem RF MAE:  {deepchem_rf_mae:.3f}")
    print(f"DeepChem RF RMSE: {deepchem_rf_rmse:.3f}")
    print(f"DeepChem RF R2:   {deepchem_rf_r2:.3f}")
else:
    dc_rf_pred = None
    deepchem_rf_mae = np.nan
    deepchem_rf_rmse = np.nan
    deepchem_rf_r2 = np.nan

In [ ]:
if HAS_DEEPCHEM:
    # Plot DeepChem RF predictions on the same test molecules.
    plt.figure(figsize=(7, 5))
    plt.scatter(y_test_true, dc_rf_pred, alpha=0.7)

    low = min(y_test_true.min(), dc_rf_pred.min())
    high = max(y_test_true.max(), dc_rf_pred.max())
    plt.plot([low, high], [low, high], linestyle="--")

    plt.xlabel("Observed logS")
    plt.ylabel("Predicted logS")
    plt.title("DeepChem RF regression performance")
    plt.show()
else:
    print("DeepChem plot skipped because DeepChem is not installed.")

### Exercise — Change DeepChem fingerprint size

Change the DeepChem circular fingerprint size and retrain the random forest. Compare the error with the default 1024-bit representation.

In [ ]:
RUN_THIS_EXERCISE = False

if RUN_THIS_EXERCISE and HAS_DEEPCHEM:
    # TODO: choose a fingerprint size.
    exercise_dc_size = ____

    # TODO: create a DeepChem circular fingerprint featuriser.
    exercise_featurizer = dc.feat.CircularFingerprint(size=____, radius=____)

    # TODO: featurise train and test SMILES.
    exercise_train_X = np.asarray(____, dtype=np.float32)
    exercise_test_X = np.asarray(____, dtype=np.float32)

    # TODO: build DeepChem datasets.
    exercise_train_dataset = dc.data.NumpyDataset(X=____, y=____, ids=____)
    exercise_test_dataset = dc.data.NumpyDataset(X=____, y=____, ids=____)

    # TODO: train a DeepChem-wrapped random forest.
    exercise_model = SklearnModel(RandomForestRegressor(n_estimators=____, random_state=SEED, n_jobs=-1))
    exercise_model.fit(____)

    # TODO: predict and calculate metrics.
    exercise_pred = ____
    print("Exercise MAE:", ____)
    print("Exercise RMSE:", ____)

### Answer — Change DeepChem fingerprint size

In [ ]:
if HAS_DEEPCHEM:
    # Use a smaller DeepChem fingerprint for comparison.
    exercise_featurizer = dc.feat.CircularFingerprint(size=512, radius=2)

    exercise_train_X = np.asarray(exercise_featurizer.featurize(train_df["smiles"].tolist()), dtype=np.float32)
    exercise_test_X = np.asarray(exercise_featurizer.featurize(test_df["smiles"].tolist()), dtype=np.float32)

    exercise_train_dataset = dc.data.NumpyDataset(
        X=exercise_train_X,
        y=y_train_true.reshape(-1, 1),
        ids=train_df["smiles"].tolist(),
    )

    exercise_test_dataset = dc.data.NumpyDataset(
        X=exercise_test_X,
        y=y_test_true.reshape(-1, 1),
        ids=test_df["smiles"].tolist(),
    )

    exercise_dc_model = SklearnModel(
        RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)
    )

    exercise_dc_model.fit(exercise_train_dataset)
    exercise_pred = exercise_dc_model.predict(exercise_test_dataset).ravel()

    print("Exercise MAE:", mean_absolute_error(y_test_true, exercise_pred))
    print("Exercise RMSE:", np.sqrt(mean_squared_error(y_test_true, exercise_pred)))
else:
    print("Install DeepChem to run this answer.")

## 11. Enhanced model comparison

The comparison table separates regression and classification metrics. Regression models use MAE, RMSE and R2. The classification model uses accuracy, F1, ROC-AUC and PR-AUC.

In [ ]:
comparison_rows = [
    {
        "method": "Random forest baseline",
        "input": "RDKit Morgan fingerprint",
        "task": "regression",
        "target": "logS",
        "status": "run",
        "MAE": rf_mae,
        "RMSE": rf_rmse,
        "R2": rf_r2,
        "accuracy": np.nan,
        "F1": np.nan,
        "ROC_AUC": np.nan,
        "PR_AUC": np.nan,
    },
    {
        "method": "PyTorch fingerprint MLP",
        "input": "RDKit Morgan fingerprint",
        "task": "regression",
        "target": "logS",
        "status": "run",
        "MAE": mlp_mae,
        "RMSE": mlp_rmse,
        "R2": mlp_r2,
        "accuracy": np.nan,
        "F1": np.nan,
        "ROC_AUC": np.nan,
        "PR_AUC": np.nan,
    },
    {
        "method": "PyTorch fingerprint MLP",
        "input": "RDKit Morgan fingerprint",
        "task": "classification",
        "target": "median-split solubility class",
        "status": "run",
        "MAE": np.nan,
        "RMSE": np.nan,
        "R2": np.nan,
        "accuracy": clf_accuracy,
        "F1": clf_f1,
        "ROC_AUC": clf_roc_auc,
        "PR_AUC": clf_pr_auc,
    },
    {
        "method": "PyTorch Geometric GNN",
        "input": "molecular graph",
        "task": "regression",
        "target": "logS",
        "status": "run" if HAS_PYG else "dependency not installed",
        "MAE": gnn_mae,
        "RMSE": gnn_rmse,
        "R2": gnn_r2,
        "accuracy": np.nan,
        "F1": np.nan,
        "ROC_AUC": np.nan,
        "PR_AUC": np.nan,
    },
    {
        "method": "DeepChem RF",
        "input": "DeepChem CircularFingerprint",
        "task": "regression",
        "target": "logS",
        "status": "run" if HAS_DEEPCHEM else "dependency not installed",
        "MAE": deepchem_rf_mae,
        "RMSE": deepchem_rf_rmse,
        "R2": deepchem_rf_r2,
        "accuracy": np.nan,
        "F1": np.nan,
        "ROC_AUC": np.nan,
        "PR_AUC": np.nan,
    },
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

In [ ]:
# Plot regression RMSE for models that produced regression metrics.
regression_comparison = comparison_df[
    (comparison_df["task"] == "regression") & comparison_df["RMSE"].notnull()
].copy()

plt.figure(figsize=(9, 5))
plt.barh(regression_comparison["method"], regression_comparison["RMSE"])
plt.xlabel("RMSE on test logS")
plt.ylabel("Model")
plt.title("Regression model comparison")
plt.gca().invert_yaxis()
plt.show()

# Plot R2 separately because higher is better.
plt.figure(figsize=(9, 5))
plt.barh(regression_comparison["method"], regression_comparison["R2"])
plt.xlabel("R2 on test logS")
plt.ylabel("Model")
plt.title("Regression R2 comparison")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Compare all available regression predictions on the same observed targets.
plt.figure(figsize=(7, 5))
plt.scatter(y_test_true, rf_pred, alpha=0.5, label="RF")
plt.scatter(y_test_true, mlp_test_pred, alpha=0.5, label="MLP")

if gnn_test_pred is not None:
    plt.scatter(y_test_true, gnn_test_pred, alpha=0.5, label="GNN")

if dc_rf_pred is not None:
    plt.scatter(y_test_true, dc_rf_pred, alpha=0.5, label="DeepChem RF")

low = y_test_true.min()
high = y_test_true.max()
plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Observed logS")
plt.ylabel("Predicted logS")
plt.title("Regression predictions across models")
plt.legend()
plt.show()

Interpretation should focus on task type and model behaviour, not just one number. A lower RMSE is better for regression, but a stable validation curve and reasonable residual pattern also matter. For classification, regression metrics are not applicable, so classification metrics are reported separately.

### Exercise — Build your own comparison table

Add one extra row for a model you trained or for a changed hyperparameter run. Keep regression and classification metrics separate.

In [ ]:
RUN_THIS_EXERCISE = False

if RUN_THIS_EXERCISE:
    # TODO: create a new result row.
    my_result = {
        "method": ____,
        "input": ____,
        "task": ____,
        "target": ____,
        "status": ____,
        "MAE": ____,
        "RMSE": ____,
        "R2": ____,
        "accuracy": ____,
        "F1": ____,
        "ROC_AUC": ____,
        "PR_AUC": ____,
    }

    # TODO: append your row to comparison_df.
    my_comparison_df = pd.concat(
        [comparison_df, pd.DataFrame([____])],
        ignore_index=True,
    )

    # TODO: display the updated comparison table.
    display(____)

### Answer — Build your own comparison table

In [ ]:
# Add the tuned MLP exercise as an example comparison row.
with torch.no_grad():
    exercise_scaled_pred = tuned_mlp_model(X_test_tensor.to(DEVICE)).cpu().numpy()

exercise_test_pred = target_scaler.inverse_transform(exercise_scaled_pred).ravel()

my_result = {
    "method": "Tuned PyTorch fingerprint MLP",
    "input": "RDKit Morgan fingerprint",
    "task": "regression",
    "target": "logS",
    "status": "run",
    "MAE": mean_absolute_error(y_test_true, exercise_test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_true, exercise_test_pred)),
    "R2": r2_score(y_test_true, exercise_test_pred),
    "accuracy": np.nan,
    "F1": np.nan,
    "ROC_AUC": np.nan,
    "PR_AUC": np.nan,
}

my_comparison_df = pd.concat(
    [comparison_df, pd.DataFrame([my_result])],
    ignore_index=True,
)

display(my_comparison_df)

## 12. Predict new molecules

After training, the fingerprint MLP can be used as a small prediction helper. This is not a validated solubility tool; it is a workflow demonstration showing how a trained molecular model can be reused.

In [ ]:
def predict_logS_with_mlp(smiles_list, model, target_scaler, radius=2, n_bits=1024):
    rows = []

    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            rows.append({
                "smiles": smiles,
                "valid": False,
                "predicted_logS": np.nan,
            })
            continue

        fp = mol_to_morgan_fp(mol, radius=radius, n_bits=n_bits)
        fp_tensor = torch.tensor(fp.reshape(1, -1), dtype=torch.float32).to(DEVICE)

        model.eval()
        with torch.no_grad():
            scaled_pred = model(fp_tensor).cpu().numpy()

        pred_logS = target_scaler.inverse_transform(scaled_pred.reshape(-1, 1)).ravel()[0]

        rows.append({
            "smiles": smiles,
            "valid": True,
            "predicted_logS": pred_logS,
        })

    return pd.DataFrame(rows)


new_smiles = [
    "CCO",
    "CC(=O)O",
    "c1ccccc1",
    "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "invalid_smiles",
]

predict_logS_with_mlp(new_smiles, mlp_model, target_scaler)

### Exercise — Predict your own molecules

Add fragrance-like, drug-like or simple organic molecules. Use valid SMILES and compare predicted logS values.

In [ ]:
RUN_THIS_EXERCISE = False

if RUN_THIS_EXERCISE:
    # TODO: add your own SMILES strings.
    my_smiles = [
        ____,
        ____,
        ____,
    ]

    # TODO: run prediction.
    my_predictions = predict_logS_with_mlp(
        smiles_list=____,
        model=____,
        target_scaler=____,
        radius=____,
        n_bits=____,
    )

    # TODO: display predictions.
    display(____)

### Answer — Predict your own molecules

In [ ]:
# Predict several fragrance-like and drug-like molecules.
my_smiles = [
    "COc1cc(C=O)ccc1O",                    # vanillin
    "CC(C)=CCCC(C)(O)C=C",                 # linalool
    "CC1=CCC(CC1)C(=C)C",                  # limonene
    "CC(=O)OC1=CC=CC=C1C(=O)O",           # aspirin
]

my_predictions = predict_logS_with_mlp(
    smiles_list=my_smiles,
    model=mlp_model,
    target_scaler=target_scaler,
    radius=FP_RADIUS,
    n_bits=FP_BITS,
)

display(my_predictions)

## 13. Summary

This notebook compares vector-based neural networks, graph neural networks and DeepChem-based workflows on the same molecular-property task.

Main points:

- A Morgan fingerprint MLP can learn useful signal but may overfit quickly.
- Classification and regression require different metrics.
- A GNN uses molecular connectivity directly instead of fixed fingerprint bits.
- DeepChem provides chemistry-focused featurisers and model wrappers.
- Model comparison should include metrics, plots and whether each dependency section actually ran.